**PDF Toolkit**

Pipeline for PDFs - possibly to be used for Micro-film scans: **resize/enhance** pages, **merge** files, and **compress** the result.

Each section below is independent — run only the cells you need. The full pipeline example is at the bottom.

## 1. Setup

In [ ]:
!apt-get install -y ghostscript poppler-utils -q
!pip install pypdf pdf2image img2pdf pillow tqdm -q

import os, io, glob, zipfile, shutil
from pypdf import PdfReader, PdfWriter
from pdf2image import convert_from_path
from PIL import Image, ImageEnhance
import img2pdf
from tqdm import tqdm
from google.colab import drive

drive.mount('/content/drive')

## 2. Flexible input handling

Accepts a `.zip` of PDFs, a folder of PDFs, or an explicit list of PDF paths.

In [ ]:
def get_pdf_paths(source):
    """
    Accepts any of:
      - path to a .zip file containing PDFs (extracts it, returns extracted paths)
      - path to a folder containing PDFs (returns paths inside it, recursively)
      - a list of explicit PDF file paths (returned as-is)
    """
    if isinstance(source, list):
        return source

    if isinstance(source, str) and source.lower().endswith(".zip"):
        extract_dir = "/content/input_pdfs"
        os.makedirs(extract_dir, exist_ok=True)
        with zipfile.ZipFile(source, "r") as z:
            z.extractall(extract_dir)
        return sorted(glob.glob(os.path.join(extract_dir, "**", "*.pdf"), recursive=True))

    if isinstance(source, str) and os.path.isdir(source):
        return sorted(glob.glob(os.path.join(source, "**", "*.pdf"), recursive=True))

    raise ValueError("source must be a .zip path, a folder path, or a list of PDF paths")

# Examples:
# get_pdf_paths("/content/drive/MyDrive/scans/batch1.zip")
# get_pdf_paths("/content/drive/MyDrive/scans/raw_pdfs")
# get_pdf_paths(["/content/drive/MyDrive/scans/a.pdf", "/content/drive/MyDrive/scans/b.pdf"])

## 3. Resize a PDF

Rasterizes each page, resizes/enhances it (same logic as the original image-based script), and rebuilds a PDF. Works directly on PDF files rather than a folder of loose JPGs.

In [ ]:
def enhance_image(img, factor):
    return ImageEnhance.Contrast(img).enhance(factor)

def image_to_jpeg_bytes_under_limit(img, max_bytes, min_quality=10):
    """Same iterative-quality-drop logic as the original script, adapted to work in memory."""
    quality = 95
    buf = io.BytesIO()
    while quality >= min_quality:
        buf.seek(0); buf.truncate(0)
        img.save(buf, format="JPEG", quality=quality, optimize=True)
        if max_bytes is None or buf.tell() <= max_bytes:
            return buf.getvalue(), True
        quality -= 5
    return buf.getvalue(), False  # couldn't hit target even at min_quality

def resize_pdf(
    input_pdf,
    output_pdf,
    resize_percent=100,       # e.g. 50 = half size, 100 = unchanged
    dpi=300,                  # dpi used to rasterize each page before resizing
    max_page_bytes=None,      # e.g. 10*1024*1024 for a 10MB/page cap, None = no cap
    enhance_contrast=False,
    contrast_factor=1.4,      # try 1.2-1.6 for faint pencil/pale ink
):
    pages = convert_from_path(input_pdf, dpi=dpi)
    jpeg_pages = []

    for i, page in enumerate(pages):
        if resize_percent != 100:
            new_w = int(page.width * resize_percent / 100)
            new_h = int(page.height * resize_percent / 100)
            page = page.resize((new_w, new_h), Image.LANCZOS)

        if enhance_contrast:
            page = enhance_image(page, contrast_factor)

        jpeg_bytes, ok = image_to_jpeg_bytes_under_limit(page, max_page_bytes)
        if not ok:
            print(f"  {os.path.basename(input_pdf)} page {i+1}: couldn't hit size target at min quality")
        jpeg_pages.append(jpeg_bytes)

    with open(output_pdf, "wb") as f:
        f.write(img2pdf.convert(jpeg_pages))

    before = os.path.getsize(input_pdf) / (1024*1024)
    after = os.path.getsize(output_pdf) / (1024*1024)
    print(f"{os.path.basename(input_pdf)}: {len(pages)} pages, {before:.1f}MB -> {after:.1f}MB")

def batch_resize(source, output_dir, **resize_kwargs):
    os.makedirs(output_dir, exist_ok=True)
    for path in tqdm(get_pdf_paths(source)):
        out_path = os.path.join(output_dir, os.path.basename(path))
        resize_pdf(path, out_path, **resize_kwargs)

# Example:
# batch_resize(
#     "/content/drive/MyDrive/scans/batch1.zip",
#     "/content/drive/MyDrive/scans/resized",
#     resize_percent=75, max_page_bytes=8*1024*1024,
#     enhance_contrast=True, contrast_factor=1.4
# )

## 4. Merge PDFs

In [ ]:
def merge_pdfs(file_paths, output_path):
    """
    Merge multiple PDFs in the given order into one file.
    file_paths: list of paths, in the order you want them merged
    output_path: where to save the merged PDF
    """
    writer = PdfWriter()
    for path in file_paths:
        reader = PdfReader(path)
        for page in reader.pages:
            writer.add_page(page)
    with open(output_path, "wb") as f:
        writer.write(f)
    print(f"Merged {len(file_paths)} files -> {output_path}")

# Example:
# merge_pdfs(
#     ["/content/drive/MyDrive/scans/part1.pdf", "/content/drive/MyDrive/scans/part2.pdf"],
#     "/content/drive/MyDrive/scans/merged.pdf"
# )

## 5. Compress a PDF (Ghostscript)

More effective than pypdf for image-heavy scanned PDFs.

`quality` options (increasing compression, decreasing quality):
- `prepress` — 300 dpi, best quality, least compression
- `printer` — 300 dpi, good quality
- `ebook` — 150 dpi, good balance (recommended default for archives)
- `screen` — 72 dpi, smallest file, lowest quality

In [ ]:
def compress_pdf(input_path, output_path, quality="ebook"):
    cmd = (
        f'gs -sDEVICE=pdfwrite -dCompatibilityLevel=1.4 '
        f'-dPDFSETTINGS=/{quality} -dNOPAUSE -dQUIET -dBATCH '
        f'-sOutputFile="{output_path}" "{input_path}"'
    )
    os.system(cmd)

    before = os.path.getsize(input_path) / (1024 * 1024)
    after = os.path.getsize(output_path) / (1024 * 1024)
    print(f"{os.path.basename(input_path)}: {before:.1f}MB -> {after:.1f}MB "
          f"({(1 - after/before)*100:.0f}% smaller)")

def batch_compress_folder(folder_path, output_folder, quality="ebook"):
    os.makedirs(output_folder, exist_ok=True)
    pdfs = [f for f in os.listdir(folder_path) if f.lower().endswith(".pdf")]
    for f in pdfs:
        in_path = os.path.join(folder_path, f)
        out_path = os.path.join(output_folder, f)
        compress_pdf(in_path, out_path, quality)

# Example:
# compress_pdf(
#     "/content/drive/MyDrive/scans/merged.pdf",
#     "/content/drive/MyDrive/scans/merged_compressed.pdf",
#     quality="ebook"
# )

## 6. Full pipeline example

Resize/enhance every page → merge into one file → final compression pass.

Edit `source` and `work` below, then run.

In [ ]:
source = "/content/drive/MyDrive/scans/batch1.zip"   # input zip, folder, or list of paths
out = "/content/drive/MyDrive/scans"                 # output location

# 1. Resize/enhance every scanned page
batch_resize(source, f"{out}/resized", resize_percent=75,
             max_page_bytes=8*1024*1024, enhance_contrast=True, contrast_factor=1.4)

# 2. Merge the resized PDFs into one file (order matters -> sort/name accordingly)
resized_paths = get_pdf_paths(f"{out}/resized")
merge_pdfs(resized_paths, f"{out}/merged.pdf")

# 3. Final compression pass on the merged file
compress_pdf(f"{out}/merged.pdf", f"{out}/final.pdf", quality="ebook")

## Notes

- **`dpi` in `resize_pdf` matters more than `resize_percent`** for scanned PDFs — it controls how sharply each page is rasterized before any shrinking happens. If your source PDFs were scanned at a high DPI, dropping `dpi` to ~200 alone (leaving `resize_percent=100`) can shave size dramatically without a percent-based resize at all.
- Rasterizing means any embedded text layer is lost. If these PDFs are pure image scans, there's nothing to lose — but if any already have OCR text baked in, resizing will strip it.
- `max_page_bytes` is optional — leave it `None` if you'd rather control size via `resize_percent`/`dpi` alone and let Ghostscript's final pass do the heavy lifting.